In [46]:
import pandas as pd
import numpy as np

In [47]:
RAW_PATH = '../../data/historyWorkflows/raw/historyWorkflows.csv'
OUTPUT_PATH = '../../data/historyWorkflows/processed/fact_history_workflows_v1.csv'

In [48]:
df = pd.read_csv(RAW_PATH)
print(f"Shape: {df.shape}")
df.head(3)

Shape: (27542, 14)


,timestamp,name,type,id,initiatorPersonName,cancelPersonName,room,cancelTimestamp,activationTime,color,subSystem,ward,scope,state
0,2024-01-31T03:44:05Z,Presence,Presence,283574,451e29e614ad790e1ef2244cf61bbd2d4a1257407ac307...,451e29e614ad790e1ef2244cf61bbd2d4a1257407ac307...,K 220,2024-01-31T03:44:06Z,10000000,#6EB221,NaN,2V ZONE 2A,NaN,NaN
1,2024-01-31T04:02:12Z,Presence,Presence,283578,92fc087f2e0d82427cdb4866c1de2b02bdd484efbda259...,92fc087f2e0d82427cdb4866c1de2b02bdd484efbda259...,ZITRUIMTE GANG203,2024-01-31T04:02:13Z,10000000,#6EB221,NaN,2V ZONE 2B,NaN,NaN
2,2024-01-31T04:06:08Z,Presence,Presence,283580,92fc087f2e0d82427cdb4866c1de2b02bdd484efbda259...,92fc087f2e0d82427cdb4866c1de2b02bdd484efbda259...,K 223,2024-01-31T04:06:09Z,10000000,#6EB221,NaN,2V ZONE 2A,NaN,NaN


In [49]:
# Check missingness — scope/state are ~97% missing, color/subSystem ~2% missing
(df.isnull().sum() / len(df) * 100).round(1)

timestamp               0.0
name                    0.0
type                    0.0
id                      0.0
initiatorPersonName     0.0
cancelPersonName        0.0
room                    0.0
cancelTimestamp         0.0
activationTime          0.0
color                   1.5
subSystem               2.3
ward                    1.5
scope                  97.5
state                  97.5
dtype: float64

In [50]:
# Drop low-value / high-missingness columns
df = df.drop(columns=['scope', 'state', 'color', 'subSystem'])

print("name values:", df['name'].value_counts().to_dict())
print("type values:", df['type'].value_counts().to_dict())

name values: {'Presence': 26594, 'Occupied': 948}
type values: {'Presence': 26594, 'State': 948}


In [51]:
# Parse timestamps — some rows have sub-second precision or date 0001-01-01 (treat as NaT)
df['timestamp_utc'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
df['cancel_timestamp_utc'] = pd.to_datetime(df['cancelTimestamp'], utc=True, errors='coerce')

# Duration of presence/occupied event in seconds
df['duration_sec'] = (df['cancel_timestamp_utc'] - df['timestamp_utc']).dt.total_seconds()

print("NaT timestamps:", df['timestamp_utc'].isna().sum())
print("NaT cancel timestamps:", df['cancel_timestamp_utc'].isna().sum())
print()
df['duration_sec'].describe().round(1)

NaT timestamps: 947
NaT cancel timestamps: 1500



count     26042.0
mean        866.0
std        6552.6
min           0.0
25%           2.0
50%         226.0
75%         864.0
max      513653.0
Name: duration_sec, dtype: float64

In [52]:
# Date/time dimensions aligned with historyCalls fact table
df['workflow_date'] = df['timestamp_utc'].dt.strftime('%d/%m/%Y')
df['workflow_hour'] = df['timestamp_utc'].dt.hour
df['workflow_dayofweek'] = df['timestamp_utc'].dt.dayofweek  # 0=Monday

def assign_shift(hour):
    if pd.isna(hour):
        return 'unknown'
    if 6 <= hour < 14:
        return 'morning'
    elif 14 <= hour < 22:
        return 'afternoon'
    else:
        return 'night'

df['shift'] = df['workflow_hour'].apply(assign_shift)

print("shift distribution:")
print(df['shift'].value_counts().to_dict())

shift distribution:
{'morning': 11497, 'afternoon': 8325, 'night': 6773, 'unknown': 947}


In [53]:
# Rename columns to snake_case, consistent with historyCalls
df = df.rename(columns={
    'id': 'workflow_id',
    'name': 'workflow_name',
    'type': 'workflow_type',
    'initiatorPersonName': 'initiator_person',
    'cancelPersonName': 'cancel_person',
    'room': 'room_name',
    'ward': 'ward_name',
    'activationTime': 'activation_time_min',
})

# Convert activation_time from milliseconds to minutes
df['activation_time_min'] = (df['activation_time_min'] / 60000).round(2)

# Drop original raw timestamp strings (keep parsed UTC versions)
df = df.drop(columns=['timestamp', 'cancelTimestamp'])

# Final column order
COLS = [
    'workflow_id', 'timestamp_utc', 'cancel_timestamp_utc',
    'workflow_date', 'workflow_hour', 'workflow_dayofweek', 'shift',
    'workflow_name', 'workflow_type',
    'ward_name', 'room_name',
    'initiator_person', 'cancel_person',
    'activation_time_min', 'duration_sec',
]
df = df[COLS]
df.head(3)

,workflow_id,timestamp_utc,cancel_timestamp_utc,workflow_date,workflow_hour,workflow_dayofweek,shift,workflow_name,workflow_type,ward_name,room_name,initiator_person,cancel_person,activation_time_min,duration_sec
0,283574,2024-01-31 03:44:05+00:00,2024-01-31 03:44:06+00:00,31/01/2024,3.0,2.0,night,Presence,Presence,2V ZONE 2A,K 220,451e29e614ad790e1ef2244cf61bbd2d4a1257407ac307...,451e29e614ad790e1ef2244cf61bbd2d4a1257407ac307...,166.67,1.0
1,283578,2024-01-31 04:02:12+00:00,2024-01-31 04:02:13+00:00,31/01/2024,4.0,2.0,night,Presence,Presence,2V ZONE 2B,ZITRUIMTE GANG203,92fc087f2e0d82427cdb4866c1de2b02bdd484efbda259...,92fc087f2e0d82427cdb4866c1de2b02bdd484efbda259...,166.67,1.0
2,283580,2024-01-31 04:06:08+00:00,2024-01-31 04:06:09+00:00,31/01/2024,4.0,2.0,night,Presence,Presence,2V ZONE 2A,K 223,92fc087f2e0d82427cdb4866c1de2b02bdd484efbda259...,92fc087f2e0d82427cdb4866c1de2b02bdd484efbda259...,166.67,1.0


In [54]:
print(f"Final shape: {df.shape}")
df.info()

Final shape: (27542, 15)
<class 'pandas.DataFrame'>
RangeIndex: 27542 entries, 0 to 27541
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   workflow_id           27542 non-null  str                
 1   timestamp_utc         26595 non-null  datetime64[us, UTC]
 2   cancel_timestamp_utc  26042 non-null  datetime64[us, UTC]
 3   workflow_date         26595 non-null  str                
 4   workflow_hour         26595 non-null  float64            
 5   workflow_dayofweek    26595 non-null  float64            
 6   shift                 27542 non-null  str                
 7   workflow_name         27542 non-null  str                
 8   workflow_type         27542 non-null  str                
 9   ward_name             27128 non-null  str                
 10  room_name             27542 non-null  str                
 11  initiator_person      27542 non-null  str            

In [45]:
import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")

PermissionError: [Errno 13] Permission denied: '../../data/historyWorkflows/processed/fact_history_workflows_v1.csv'